# Âge, groupe et montant d'achat

Nous conservons la structure du projet gaussien du 29 septembre.
La classe `GaussianMixture` reste disponible et inchangée. Pour ce
nouveau jeu de données, nous suivons cinq étapes simples :

1. créer la colonne `group`, avec A ou B ;
2. appeler `add_age()` : les âges diffèrent entre A et B ;
3. appeler `add_purchase_amount()` : le montant dépend de l'âge,
   du groupe et d'un bruit aléatoire.
4. appeler `add_checkout_wait_minutes()` : même règle aléatoire
   pour toutes les transactions ;
5. appeler `add_distance_to_store_km()` : la distribution dépend
   du magasin A ou B, sans entrer dans la règle du montant.

Règle de simulation :

`purchase_amount = round(baseline + age_effect × age + group_effect × I(group=B) + noise, 1)`

Ici, `I(group=B)` vaut 1 pour B et 0 pour A.

Dans la narration, A est un magasin de centre-ville et B un magasin
périurbain. Il s'agit d'une règle de simulation, pas d'une propriété
générale de ces types de magasins.

## 1. Créer les lignes A/B

**Avant d'exécuter :** prédisez la colonne et les effectifs du
dataframe créé avec `n=2000` et `weight_a=0.6`.

<!-- response:base_contract -->
### Votre réponse — contrat du jeu de données initial

Nommez la colonne initiale et les effectifs attendus dans A et B.

> Écrivez votre réponse ici.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

project_root = Path("../lab/gaussian_project_solution").resolve()
if not project_root.exists():
    project_root = Path("gaussian_project_solution").resolve()
if not project_root.exists():
    raise FileNotFoundError(
        "Placez le notebook à côté du dossier lab ou du projet extrait."
    )
sys.path.insert(0, str(project_root))

from analysis import (
    compare_correlations,
    pearson_association,
    purchase_gap_decomposition,
    purchase_mean_test,
)
from distributions.age import add_age
from distributions.purchase_dataset import (
    AGE_EFFECT,
    BASELINE,
    GROUP_EFFECT,
    add_checkout_wait_minutes,
    add_distance_to_store_km,
    add_purchase_amount,
    make_groups,
)

group_samples = make_groups(n=2000, weight_a=0.6)
group_samples.head()

FileNotFoundError: Placez le notebook à côté du dossier lab ou du projet extrait.

In [ ]:
print(group_samples.dtypes)
print("\nEffectifs :")
print(group_samples["group"].value_counts().sort_index())

## 2. Ajouter l'âge avec `add_age()`

La fonction utilise une moyenne de 40 ans dans A et de 48 ans dans
B, avec le même écart-type de 8 ans. Elle retourne une copie du
dataframe et ne modifie pas son argument.

<!-- response:age_prediction -->
### Votre réponse — prédiction des âges

Prédisez le signe et l'ordre de grandeur de avg(age_B) − avg(age_A).

> Écrivez votre réponse ici.

In [ ]:
with_age = add_age(group_samples, seed=405)

pd.testing.assert_frame_equal(
    with_age[["group"]], group_samples
)
age_summary = with_age.groupby("group")["age"].agg(["count", "mean", "std"])
display(age_summary.round(3))
print(f"Écart d'âge B−A : {age_summary.loc['B', 'mean'] - age_summary.loc['A', 'mean']:.3f} ans")

<!-- response:age_result -->
### Votre réponse — résultat sur l'âge

Donnez les deux moyennes et expliquez pourquoi leur différence existe dans cette simulation.

> Écrivez votre réponse ici.

## 3. Ajouter le montant d'achat

Les paramètres utilisés sont `baseline = 80`, `age_effect = 0.5`,
`group_effect = 2` et `noise_std = 8`.

À âge égal, la moyenne attendue de B dépasse donc celle de A de
2 €. Chaque année d'âge ajoute 0,50 € à la moyenne attendue.

Chaque montant est ensuite arrondi au dixième d'euro, donc par
incréments de 0,10 €. Dans le CSV, il est écrit avec deux décimales
(`123.40`). Après lecture, pandas peut afficher `123.4` : la valeur
est la même.

<!-- response:purchase_prediction -->
### Votre réponse — prédiction des montants

Comparez les montants attendus pour A à 40 ans, B à 40 ans et B à 48 ans.

> Écrivez votre réponse ici.

In [ ]:
samples = add_purchase_amount(with_age, seed=406)

pd.testing.assert_frame_equal(
    samples[["group", "age"]], with_age
)
print(f"A, 40 ans : {BASELINE + AGE_EFFECT * 40:.1f} € attendus")
print(f"B, 40 ans : {BASELINE + AGE_EFFECT * 40 + GROUP_EFFECT:.1f} € attendus")
print(f"B, 48 ans : {BASELINE + AGE_EFFECT * 48 + GROUP_EFFECT:.1f} € attendus")
samples.head()

<!-- response:purchase_result -->
### Votre réponse — lecture de la règle

Séparez l'effet de l'âge, l'effet restant du groupe et le rôle du bruit.

> Écrivez votre réponse ici.

## 4. Comparer les moyennes de montant

H₀ : `μB − μA = 0`. H₁ : `μB − μA ≠ 0`.
L'intervalle porte sur la différence des moyennes de population.

Comme B est plus âgé en moyenne **et** conserve un effet propre de
2 €, l'écart brut réunit deux mécanismes connus par construction.

Les statistiques ci-dessous utilisent 2 000 transactions et sont
affichées à trois décimales. Cela n'augmente pas la précision de
chaque montant individuel, arrondi au dixième d'euro.

In [ ]:
group_summary = samples.groupby("group")[["age", "purchase_amount"]].agg(
    ["count", "mean", "std"]
)
mean_test = purchase_mean_test(samples)
gap_parts = purchase_gap_decomposition(
    samples,
    age_effect=AGE_EFFECT,
    group_effect=GROUP_EFFECT,
)
display(group_summary.round(3))
print(
    f"Écart de montant B−A : {mean_test['difference']:.3f} €\n"
    f"IC à 95 % : [{mean_test['confidence_low']:.3f}, "
    f"{mean_test['confidence_high']:.3f}] €\n"
    f"Valeur p : {mean_test['p_value']:.3g}\n"
    f"Part liée au profil d'âge : {gap_parts['age_profile_part']:.3f} €\n"
    f"Part restante du groupe : {gap_parts['remaining_group_part']:.3f} €\n"
    f"Résidu observé (bruit + arrondi) : "
    f"{gap_parts['observed_residual_gap']:.3f} €"
)

<!-- response:group_result -->
### Votre réponse — comparaison A/B

Donnez l'écart brut, son IC à 95 %, puis décomposez-le avec les mécanismes programmés.

> Écrivez votre réponse ici.

## 5. Observer `age` et `purchase_amount`

**Avant d'exécuter :** prédisez le sens des pentes dans A et B.
Les deux droites moyennes attendues seront-elles parallèles ?

<!-- response:scatter_prediction -->
### Votre réponse — prédiction des nuages

Prédisez la forme du nuage, les pentes et le décalage entre A et B.

> Écrivez votre réponse ici.

In [ ]:
plot_sample = samples.sample(n=700, random_state=404)
figure = px.scatter(
    plot_sample,
    x="age",
    y="purchase_amount",
    color="group",
    opacity=0.38,
    labels={"age": "Âge (ans)", "purchase_amount": "Montant d'achat (€)"},
    title="Le montant augmente avec l'âge dans A et dans B",
    color_discrete_map={"A": "#0072B2", "B": "#D55E00"},
)
ages = [18, 80]
figure.add_scatter(
    x=ages,
    y=[BASELINE + AGE_EFFECT * age for age in ages],
    mode="lines",
    name="Moyenne attendue A",
    line={"color": "#0072B2", "width": 3},
)
figure.add_scatter(
    x=ages,
    y=[BASELINE + AGE_EFFECT * age + GROUP_EFFECT for age in ages],
    mode="lines",
    name="Moyenne attendue B",
    line={"color": "#D55E00", "width": 3},
)
figure.show()

<!-- response:scatter_result -->
### Votre réponse — description des nuages

Décrivez la tendance, la dispersion, le chevauchement et le décalage vertical entre les groupes.

> Écrivez votre réponse ici.

## 6. Covariance et corrélation

La covariance garde ici l'unité `années × euros`. La corrélation de
Pearson `r` est sans unité et résume une association linéaire.

In [ ]:
display(samples[["age", "purchase_amount"]].cov().round(3))
display(samples[["age", "purchase_amount"]].corr().round(3))

## 7. Comparer A+B, A et B

Le résultat global et les résultats par groupe ne répondent pas à
la même question. Comparons les trois corrélations descriptives.

In [ ]:
correlation_table = compare_correlations(samples).reset_index()
display(
    correlation_table[["scope", "count", "correlation"]]
    .rename(columns={"count": "n", "correlation": "r"})
    .round({"n": 0, "r": 3})
)

<!-- response:scope_comparison -->
### Votre réponse — groupes et périmètres

Comparez les trois r. Pourquoi l'association reste-t-elle positive dans A et dans B ?

> Écrivez votre réponse ici.

## 8. Lire l'incertitude sans changer de périmètre

Comparons le résultat global et celui de A. Une petite valeur p ne
mesure pas l'intensité de l'association.

In [ ]:
for group in (None, "A"):
    result = pearson_association(
        samples,
        group=group,
    )
    print(
        f"{result['scope']} : r = {result['correlation']:.3f}, "
        f"IC 95 % [{result['confidence_low']:.3f}, "
        f"{result['confidence_high']:.3f}], "
        f"p = {result['p_value']:.3g}"
    )

<!-- response:formal_scope -->
### Votre réponse — incertitude et périmètre

Interprétez séparément A+B et A. Distinguez ampleur, incertitude, mécanisme simulé et causalité.

> Écrivez votre réponse ici.

## 9. Ajouter deux variables pour comparer trois mécanismes

Une ligne représente maintenant une transaction simulée :

- `checkout_wait_minutes` est tiré avec la même loi pour toutes les
  lignes, sans lire les autres colonnes ;
- `distance_to_store_km` utilise `group` : dans cette simulation,
  la clientèle de B vient en moyenne de plus loin ;
- aucune des deux variables n'entre dans la formule de
  `purchase_amount`.

**Attention :** l'absence de lien programmé ne garantit pas une
corrélation observée exactement égale à zéro.

<!-- response:context_prediction -->
### Votre réponse — prédiction des deux nouvelles variables

Prédisez les profils A/B et les corrélations avec purchase_amount dans A+B, A et B.

> Écrivez votre réponse ici.

In [ ]:
with_wait = add_checkout_wait_minutes(samples, seed=407)
extended_samples = add_distance_to_store_km(with_wait, seed=408)

pd.testing.assert_frame_equal(
    extended_samples[["group", "age", "purchase_amount"]],
    samples,
)
display(
    extended_samples.groupby("group")[[
        "checkout_wait_minutes",
        "distance_to_store_km",
    ]]
    .agg(["mean", "std"])
    .round(3)
)

## 10. Observer les trois nuages avec les mêmes repères

Les droites sont descriptives. Elles résument les données simulées ;
elles ne prouvent aucun mécanisme causal.

<!-- response:three_scatter_prediction -->
### Votre réponse — prédiction des trois nuages

Pour l'attente, la distance et l'âge, prédisez séparément la droite globale et les droites dans A et B.

> Écrivez votre réponse ici.

In [ ]:
predictor_labels = {
    "checkout_wait_minutes": "Temps d'attente en caisse (minutes)",
    "distance_to_store_km": "Distance domicile–magasin (km)",
    "age": "Âge (ans)",
}
plot_sample = extended_samples.sample(n=700, random_state=404)

for predictor, x_label in predictor_labels.items():
    figure = px.scatter(
        plot_sample,
        x=predictor,
        y="purchase_amount",
        color="group",
        opacity=0.32,
        labels={predictor: x_label, "purchase_amount": "Montant d'achat (€)"},
        title=f"{x_label} et montant d'achat",
        color_discrete_map={"A": "#0072B2", "B": "#D55E00"},
    )
    for group, color in (("A", "#0072B2"), ("B", "#D55E00")):
        group_rows = extended_samples.loc[extended_samples["group"] == group]
        slope, intercept = np.polyfit(
            group_rows[predictor],
            group_rows["purchase_amount"],
            1,
        )
        x_line = np.array([group_rows[predictor].min(), group_rows[predictor].max()])
        figure.add_scatter(
            x=x_line,
            y=intercept + slope * x_line,
            mode="lines",
            name=f"Droite descriptive {group}",
            line={"color": color, "width": 3},
        )
    pooled_slope, pooled_intercept = np.polyfit(
        extended_samples[predictor],
        extended_samples["purchase_amount"],
        1,
    )
    pooled_x = np.array([
        extended_samples[predictor].min(),
        extended_samples[predictor].max(),
    ])
    figure.add_scatter(
        x=pooled_x,
        y=pooled_intercept + pooled_slope * pooled_x,
        mode="lines",
        name="Droite descriptive A+B",
        line={"color": "#102A43", "width": 3, "dash": "dash"},
    )
    figure.show()

<!-- response:three_scatter_result -->
### Votre réponse — lecture des trois nuages

Décrivez les pentes globales et internes. Reliez chacune à la fonction qui a généré la variable.

> Écrivez votre réponse ici.

## 11. Comparer les corrélations sur les trois périmètres

Une association globale de la distance peut apparaître parce que
`group` déplace à la fois la distance et le montant. Cela ne signifie
pas que parcourir davantage de kilomètres augmente l'achat.

In [ ]:
predictor_correlations = pd.concat(
    [
        compare_correlations(extended_samples, predictor=predictor)
        .reset_index()
        .assign(predictor=predictor)
        for predictor in predictor_labels
    ],
    ignore_index=True,
)
display(
    predictor_correlations.pivot(
        index="predictor",
        columns="scope",
        values="correlation",
    )[["A+B", "A", "B"]].round(3)
)

<!-- response:context_correlation_result -->
### Votre réponse — trois mécanismes

Classez les variables en lien direct programmé, aucune association programmée et association globale de composition.

> Écrivez votre réponse ici.

## Conclusion à rédiger

Votre conclusion doit contenir :

1. l'écart d'âge B−A ;
2. l'écart de `purchase_amount` B−A et son IC à 95 % ;
3. la comparaison entre `r_A+B`, `r_A` et `r_B` ;
4. la part liée au profil d'âge et la part restante du groupe ;
5. le mécanisme de `checkout_wait_minutes` ;
6. la différence entre le résultat global et les résultats internes
   pour `distance_to_store_km` ;
7. une limite sur la causalité.

<!-- response:final_conclusion -->
### Votre réponse — conclusion en sept éléments

Rédigez une conclusion autonome qui reprend les sept éléments demandés.

> Écrivez votre réponse ici.